# Basic Self-Attention (TensorFlow) - End-to-End Pipeline
**Date**: 2026-05-31  
**Objective**: Implement tokenization, preprocessing, and a basic multi-head self-attention classifier using TensorFlow.

## 1) Imports and Setup

In [ ]:
from __future__ import annotations

import math
import os
import random
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def parse_use_gpu_flag(raw_value: str) -> bool:
    normalized = raw_value.strip().lower()
    return normalized not in {"0", "false", "no", "off"}

def load_runtime_env() -> None:
    env_files = [
        Path("configs/runtime.env"),
        Path("configs/runtime.env.example"),
        Path("../configs/runtime.env"),
        Path("../configs/runtime.env.example"),
    ]
    for env_file in env_files:
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            break

def register_local_module_path() -> None:
    candidates = [Path("."), Path("self-attention-variants"), Path("../self-attention-variants")]
    for candidate in candidates:
        if (candidate / "shared_text_preprocessing.py").exists():
            resolved = str(candidate.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            return
    raise FileNotFoundError("Could not locate shared_text_preprocessing.py")

load_runtime_env()
register_local_module_path()
USE_GPU = parse_use_gpu_flag(os.getenv("USE_GPU", "1"))
runtime_device = "cuda" if USE_GPU and bool(tf.config.list_physical_devices("GPU")) else "cpu"
print(f"USE_GPU={int(USE_GPU)} | runtime_device={runtime_device}")

from shared_text_preprocessing import PreprocessingConfig, prepare_text_data

## 2) Configuration and Constants

In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    num_samples: int = 256
    seq_len: int = 24
    d_model: int = 32
    num_heads: int = 4
    learning_rate: float = 0.05
    epochs: int = 120


CFG = ExperimentConfig()
CFG

## 3) Data Loading

In [ ]:
prep_cfg = PreprocessingConfig(
    num_samples=CFG.num_samples,
    seq_len=CFG.seq_len,
    max_vocab_size=2048,
    train_ratio=0.8,
)
data = prepare_text_data(prep_cfg, seed=SEED)
len(data.texts_train), len(data.texts_test), data.y_train.shape, data.y_test.shape

## 4) EDA

In [ ]:
train_lengths = np.array([len(text.split()) for text in data.texts_train], dtype=np.int64)
class_counts = np.bincount(np.concatenate([data.y_train, data.y_test]), minlength=2)
print("Class counts:", class_counts.tolist())
print("Train token-length mean/std:", float(train_lengths.mean()), float(train_lengths.std()))
print("Sample training sentence:", data.texts_train[0])

## 5) Preprocessing / Feature Engineering

In [ ]:
embedding = tf.keras.layers.Embedding(
    input_dim=len(data.vocab),
    output_dim=CFG.d_model,
    embeddings_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.2, seed=SEED),
)

x_train = embedding(tf.convert_to_tensor(data.train_token_ids, dtype=tf.int32))
x_test = embedding(tf.convert_to_tensor(data.test_token_ids, dtype=tf.int32))
print({"vocab_size": len(data.vocab), "x_train_shape": x_train.shape, "x_test_shape": x_test.shape})

## 6) Model Definition

In [ ]:
def init_projection_weights(d_model: int) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    w_q = tf.random.normal((d_model, d_model), stddev=0.2, seed=SEED)
    w_k = tf.random.normal((d_model, d_model), stddev=0.2, seed=SEED + 1)
    w_v = tf.random.normal((d_model, d_model), stddev=0.2, seed=SEED + 2)
    return w_q, w_k, w_v

def apply_qkv_projection(
    x: tf.Tensor,
    w_q: tf.Tensor,
    w_k: tf.Tensor,
    w_v: tf.Tensor,
) -> tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
    return tf.linalg.matmul(x, w_q), tf.linalg.matmul(x, w_k), tf.linalg.matmul(x, w_v)

def multi_head_attention(q: tf.Tensor, k: tf.Tensor, v: tf.Tensor, num_heads: int) -> tuple[tf.Tensor, tf.Tensor]:
    batch = tf.shape(q)[0]
    seq_len = tf.shape(q)[1]
    d_model = q.shape[-1]
    if d_model is None:
        raise ValueError("d_model must be statically known for this demo.")
    if d_model % num_heads != 0:
        raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
    head_dim = d_model // num_heads

    qh = tf.transpose(tf.reshape(q, (batch, seq_len, num_heads, head_dim)), perm=(0, 2, 1, 3))
    kh = tf.transpose(tf.reshape(k, (batch, seq_len, num_heads, head_dim)), perm=(0, 2, 1, 3))
    vh = tf.transpose(tf.reshape(v, (batch, seq_len, num_heads, head_dim)), perm=(0, 2, 1, 3))

    scores = tf.linalg.matmul(qh, kh, transpose_b=True) / math.sqrt(float(head_dim))
    weights = tf.nn.softmax(scores, axis=-1)
    out = tf.linalg.matmul(weights, vh)
    out = tf.reshape(tf.transpose(out, perm=(0, 2, 1, 3)), (batch, seq_len, d_model))
    return out, weights

def pooled_features(attn_out: tf.Tensor) -> tf.Tensor:
    return tf.reduce_mean(attn_out, axis=1)

## 7) Training

In [ ]:
w_q, w_k, w_v = init_projection_weights(CFG.d_model)
q_train, k_train, v_train = apply_qkv_projection(x_train, w_q, w_k, w_v)
q_test, k_test, v_test = apply_qkv_projection(x_test, w_q, w_k, w_v)

attn_train, train_weights = multi_head_attention(q_train, k_train, v_train, CFG.num_heads)
x_feat_train = tf.stop_gradient(pooled_features(attn_train))

head = tf.keras.layers.Dense(1)
optimizer = tf.keras.optimizers.SGD(learning_rate=CFG.learning_rate)
y_train_t = tf.convert_to_tensor(data.y_train.reshape(-1, 1), dtype=tf.float32)

for _ in range(CFG.epochs):
    with tf.GradientTape() as tape:
        logits = head(x_feat_train)
        loss = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y_train_t, logits=logits))
    grads = tape.gradient(loss, head.trainable_variables)
    optimizer.apply_gradients(zip(grads, head.trainable_variables))

print("Training finished.")

## 8) Evaluation and Metrics

In [ ]:
attn_test, test_weights = multi_head_attention(q_test, k_test, v_test, CFG.num_heads)
x_feat_test = pooled_features(attn_test)
logits_test = head(x_feat_test)
probs_test = tf.math.sigmoid(logits_test)
preds_test = tf.cast(probs_test >= 0.5, tf.int64).numpy().reshape(-1)
accuracy = float((preds_test == data.y_test).mean())
metrics = {"accuracy": accuracy}
metrics

## 9) Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Class 0", "Class 1"], class_counts, color=["#4C78A8", "#F58518"])
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("count")

pred_counts = np.bincount(preds_test, minlength=2)
axes[1].bar(["Pred 0", "Pred 1"], pred_counts, color=["#54A24B", "#E45756"])
axes[1].set_title(f"Prediction Distribution | acc={metrics['accuracy']:.3f}")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

## 10) Summary
- Loaded a synthetic raw-text corpus and labels using a shared preprocessing module.
- Converted token IDs to embeddings and trained a TensorFlow multi-head self-attention classifier.
- Evaluated test accuracy and visualized label and prediction distributions.